In [1]:
from langgraph.graph import StateGraph , START , END
from typing import TypedDict
from langchain_huggingface import HuggingFacePipeline , ChatHuggingFace
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

e:\langgarph\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
LOCAL_MODEL_PATH = r"C:\Users\dell\.cache\huggingface\hub\models--TinyLlama--TinyLlama-1.1B-Chat-v1.0\snapshots\fe8a4ea1ffedaf415f4da2f062534de366a451e6"

tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_PATH)

model = AutoModelForCausalLM.from_pretrained(
    LOCAL_MODEL_PATH,
    torch_dtype="auto"
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    do_sample=True,
    temperature=0.7
)

llm = HuggingFacePipeline(pipeline=pipe)
chat_model = ChatHuggingFace(llm=llm)

`torch_dtype` is deprecated! Use `dtype` instead!
Device set to use cpu


In [48]:
class LLM_state(TypedDict):
    quesion: str
    answer : str

In [49]:
def generate_ans(state:LLM_state) -> LLM_state:
    quesion = state["quesion"]

    prompt = f"find the short ans on {quesion}"

    answer = chat_model.invoke(prompt)

    state["answer"] = answer

    return state

In [50]:
graph = StateGraph(LLM_state)

graph.add_node("generate_ans",generate_ans)

graph.add_edge(START,"generate_ans")
graph.add_edge("generate_ans",END)

workflow = graph.compile()

In [51]:
intial_state = {'quesion': 'How far is moon from the earth?'}

final_state = workflow.invoke(intial_state)

print(final_state)

{'quesion': 'How far is moon from the earth?', 'answer': AIMessage(content='<|user|>\nfind the short ans on How far is moon from the earth?</s>\n<|assistant|>\nTo find the short answer for "How far is the moon from the earth?" you can use the distance formula:\n\ndistance = (apoapsis + perigee) * (1 + e) / 2\n\nwhere:\n- apoapsis: the farthest point from the sun reached by the moon during its elliptical orbit\n- perigee: the closest point to the earth reached by the moon during its elliptical orbit\n- e: the eccentricity of the moon\'s orbit\n\nIn this case, the short answer is:\n\n- distance = (apoapsis + perigee) * (1 + e) / 2 = 354,500 km (147,000 miles)\n\nNote that this distance assumes the moon orbits the earth at a distance of 354,500 kilometers (213,750 miles) from the earth\'s surface.', additional_kwargs={}, response_metadata={}, id='run--019cc2a3-0b9c-7b51-bd80-3a7570ff9368-0')}
